# Meme Organizer: 03. Hybrid Retrieval & Cross-Encoder Re-ranking
В этом блокноте финализируется **текстовый домен поиска**:
1. **First-Stage Late Fusion**:
   - **RRF (Reciprocal Rank Fusion)**: бескомпромиссное объединение рангов BM25 и Fine-tuned E5 без параметров ($k=60$).
   - **Optuna Convex Score Combination**: калибровка веса $\alpha \in [0, 1]$ на валидационном сплите (`val`) с последующей честной оценкой на тесте (`test`).
2. **Second-Stage Re-ranking**:
   - Отбор кандидатов (топ-20) первой стадией и повторное ранжирование легковесным ONNX INT8 Cross-Encoder (`rubert-tiny-bilingual-nli`, размер 11.3 МБ).
   - Оценка выигрыша по качеству ($Recall@1$, $Recall@5$, $MRR@5$) и накладных расходов по задержке ($ms$).


In [1]:
import json
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd

from meme_organizer.data_splits.splitter import DatasetSplitter
from meme_organizer.evaluation.metrics import RetrievalEvaluator
from meme_organizer.retrieval.adapters.hybrid import HybridRetrieverAdapter
from meme_organizer.retrieval.pipeline import RetrievalPipeline

GOLD_DATASET_PATH = Path("../data/gold/gold_dataset.jsonl")
SPLIT_PATH = Path("../data/splits/splits_v1.json")

2026/11/24 21:16:06 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at /home/alex/Projects/meme-organizer/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/mlflow-trace-instrumentation/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


## 1. Проверка золотых сплитов

In [2]:
with open(SPLIT_PATH) as f:
    split_info = json.load(f)

print(f"Корпус мемов: {len(split_info['corpus_hashes'])} документов")
print(f"Train пары:   {len(split_info['train_pairs'])} запросов")
print(f"Val пары:     {len(split_info['val_pairs'])} запросов")
print(f"Test пары:    {len(split_info['test_pairs'])} запросов")

Корпус мемов: 318 документов
Train пары:   1021 запросов
Val пары:     138 запросов
Test пары:    153 запросов


## 2. Эксперимент 1: Late Fusion via Reciprocal Rank Fusion (RRF)
RRF стабилен к разнице шкал и распределений плотных и разреженных скоров:
$$RRF(d) = \sum_{m} \frac{1}{60 + rank_m(d)}$$


In [3]:
pipeline = RetrievalPipeline()
adapter_rrf = HybridRetrieverAdapter(mode="rrf", model_name="hybrid-rrf")

metrics_rrf, run_id_rrf = pipeline.run(
    adapter=adapter_rrf,
    gold_dataset_path=GOLD_DATASET_PATH,
    split_path=SPLIT_PATH,
    split_name="test",
)

pd.DataFrame(
    [
        {
            "Model": "Hybrid RRF (BM25 + E5-FT)",
            "Recall@1": metrics_rrf.recall_at_1,
            "Recall@5": metrics_rrf.recall_at_5,
            "MRR@5": metrics_rrf.mrr_at_5,
            "NDCG@5": metrics_rrf.ndcg_at_5,
            "MLflow Run ID": run_id_rrf,
        }
    ]
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

🏃 View run hybrid-rrf_test at: http://localhost:5000/#/experiments/4/runs/59d63186c9fa426e9b3ae7b92fb288e0
🧪 View experiment at: http://localhost:5000/#/experiments/4


,Model,Recall@1,Recall@5,MRR@5,NDCG@5,MLflow Run ID
0,Hybrid RRF (BM25 + E5-FT),0.607843,0.718954,0.648039,0.665695,59d63186c9fa426e9b3ae7b92fb288e0


## 3. Эксперимент 2: Convex Score Combination с подбором веса $\alpha$ через Optuna
Подбор веса $\alpha$ происходит **строго на валидационном сплите** (`val_pairs`), после чего замороженный вес применяется к тесту (`test_pairs`).


In [4]:
adapter_optuna = HybridRetrieverAdapter(mode="optuna", model_name="hybrid-optuna")

metrics_optuna, run_id_optuna = pipeline.run(
    adapter=adapter_optuna,
    gold_dataset_path=GOLD_DATASET_PATH,
    split_path=SPLIT_PATH,
    split_name="test",
)

print(f"Подобранный вес alpha (Dense vs BM25): {adapter_optuna.alpha:.4f}")
pd.DataFrame(
    [
        {
            "Model": f"Hybrid Convex (alpha={adapter_optuna.alpha:.2f})",
            "Recall@1": metrics_optuna.recall_at_1,
            "Recall@5": metrics_optuna.recall_at_5,
            "MRR@5": metrics_optuna.mrr_at_5,
            "NDCG@5": metrics_optuna.ndcg_at_5,
            "MLflow Run ID": run_id_optuna,
        }
    ]
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

🏃 View run hybrid-optuna_test at: http://localhost:5000/#/experiments/4/runs/da84cc678e3c4d86b80433d63d95c623
🧪 View experiment at: http://localhost:5000/#/experiments/4
Подобранный вес alpha (Dense vs BM25): 0.5000


,Model,Recall@1,Recall@5,MRR@5,NDCG@5,MLflow Run ID
0,Hybrid Convex (alpha=0.50),0.660131,0.764706,0.701961,0.717735,da84cc678e3c4d86b80433d63d95c623


## 4. Эксперимент 3: Two-Stage Retrieval с Cross-Encoder Re-ranking
Отбираем топ-20 кандидатов первой стадией и реранжируем их с помощью `rubert-tiny-bilingual-nli` (INT8 ONNX).


In [5]:
adapter_ce = HybridRetrieverAdapter(
    mode="rerank_ce", model_name="hybrid-rerank-ce", rerank_top_k=20, alpha=adapter_optuna.alpha
)

metrics_ce, run_id_ce = pipeline.run(
    adapter=adapter_ce,
    gold_dataset_path=GOLD_DATASET_PATH,
    split_path=SPLIT_PATH,
    split_name="test",
)

pd.DataFrame(
    [
        {
            "Model": "Two-Stage (Hybrid + CE Reranker)",
            "Recall@1": metrics_ce.recall_at_1,
            "Recall@5": metrics_ce.recall_at_5,
            "MRR@5": metrics_ce.mrr_at_5,
            "NDCG@5": metrics_ce.ndcg_at_5,
            "MLflow Run ID": run_id_ce,
        }
    ]
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

🏃 View run hybrid-rerank-ce_test at: http://localhost:5000/#/experiments/4/runs/20c496e689074b2885792c132c8df9b1
🧪 View experiment at: http://localhost:5000/#/experiments/4


,Model,Recall@1,Recall@5,MRR@5,NDCG@5,MLflow Run ID
0,Two-Stage (Hybrid + CE Reranker),0.405229,0.69281,0.50915,0.554715,20c496e689074b2885792c132c8df9b1


## 5. Итоговый лидерборд текстового поиска (MLflow Comparison)

In [6]:
client = mlflow.tracking.MlflowClient(tracking_uri="http://localhost:5000")
exp = client.get_experiment_by_name("meme-organizer-retrieval")
runs = client.search_runs(experiment_ids=[exp.experiment_id], order_by=["attributes.start_time ASC"])

data = []
for r in runs:
    name = r.data.tags.get("mlflow.runName", "unnamed")
    m = r.data.metrics
    data.append(
        {
            "Model": name,
            "Recall@1": m.get("recall_at_1", 0),
            "Recall@5": m.get("recall_at_5", 0),
            "MRR@5": m.get("mrr_at_5", 0),
            "NDCG@5": m.get("ndcg_at_5", 0),
            "P95 Latency (ms)": m.get("cpu_latency_p95_ms", 0),
            "Model Size (MB)": m.get("model_size_mb", 0),
            "Eval Time (s)": m.get("eval_inference_time_sec", 0),
        }
    )

df_leaderboard = pd.DataFrame(data)
df_leaderboard.sort_values(by="MRR@5", ascending=False).reset_index(drop=True)

,Model,Recall@1,Recall@5,MRR@5,NDCG@5,P95 Latency (ms),Model Size (MB),Eval Time (s)
0,hybrid-optuna_test,0.660131,0.764706,0.701961,0.717735,13.00,112.62,2.740
1,hybrid-optuna_test,0.660131,0.764706,0.701961,0.717735,5.93,112.62,2.642
2,hybrid-optuna_test,0.660131,0.764706,0.701961,0.717735,9.59,0.00,2.555
3,bm25-baseline_test,0.607843,0.725490,0.654357,0.672219,0.00,0.10,0.019
4,hybrid-rrf_test,0.607843,0.718954,0.648039,0.665695,6.00,112.62,2.731
5,hybrid-rrf_test,0.607843,0.718954,0.648039,0.665695,6.78,0.00,2.707
6,hybrid-rrf_test,0.607843,0.718954,0.648039,0.665695,5.90,0.00,2.777
7,hybrid-rrf_test,0.607843,0.718954,0.648039,0.665695,6.90,112.62,2.787
8,multilingual-e5-small-finetuned_test,0.568627,0.666667,0.608061,0.622898,10.63,112.52,3.532
9,multilingual-e5-small_test,0.562092,0.666667,0.603159,0.619126,6.52,112.52,2.616
